# Transforming the age groups and sex dataset

This notebook transforms the cleaned age groups and sex dataset `ibe030-migration-by-age-groups-and-sex-2016-2025-interim.csv` into an analysis-ready wide format dataset. The goal is to pivot `Rādītāji` column into two, add a new column named `Migrācijas saldo` (`Iebrauca Latvijā - Izbrauca no Latvijas`) and save the dataset in `data/processed` for further work.

In [1]:
import pandas as pd

age_groups_and_sex_data = pd.read_csv("../data/interim/ibe030-migration-by-age-groups-and-sex-2016-2025-interim.csv")

age_groups_and_sex_data.head()

,Rādītāji,Vecuma grupa,Dzimums,Gads,Skaits
0,Iebrauca Latvijā,Pavisam,Pavisam,2016,8345
1,Iebrauca Latvijā,Pavisam,Vīrieši,2016,4740
2,Iebrauca Latvijā,Pavisam,Sievietes,2016,3605
3,Iebrauca Latvijā,0–4 gadi,Pavisam,2016,976
4,Iebrauca Latvijā,0–4 gadi,Vīrieši,2016,538


The required libraries are imported and the cleaned age groups and sex dataset is loaded.
The first rows confirm that the dataset was loaded as intended from the cleaning work, with all expected columns and values.

In [2]:
age_groups_and_sex_pivoted = age_groups_and_sex_data.pivot_table(
    index=["Vecuma grupa", "Dzimums", "Gads"],
    columns="Rādītāji",
    values="Skaits",
    aggfunc="sum"
).reset_index()

age_groups_and_sex_pivoted.columns.name = None

age_groups_and_sex_pivoted.head()

,Vecuma grupa,Dzimums,Gads,Iebrauca Latvijā,Izbrauca no Latvijas
0,0–4 gadi,Pavisam,2016,976,727
1,0–4 gadi,Pavisam,2017,747,677
2,0–4 gadi,Pavisam,2018,670,649
3,0–4 gadi,Pavisam,2019,655,511
4,0–4 gadi,Pavisam,2020,484,467


The dataset is transformed from long to wide format using `.pivot_table()`.
`Vecuma grupa`, `Dzimums` and `Gads` are used as identifier columns, so each row represents one age group, sex, and year. The values in `Rādītāji` column are converted into two separate columns that were the values of the column: `Iebrauca Latvijā` and `Izbrauca no Latvijas`. The migration counts from `Skaits` column are placed into these new columns.

`aggfunc="sum"` is used to aggregate values if more than one value exists for the same combination of age group, sex, year, and migration indicator. It was previously confirmed that no duplicate rows exist, so it is kept as the default aggregation option if new entries are added to the dataset in the future.

`.reset_index()` moves `Vecuma grupa`, `Dzimums`, and `Gads` back into normal columns after pivoting.

`age_groups_and_sex_pivoted.columns.name = None` removes `Rādītāji` column name that pandas keeps above the new columns after pivoting. It does not remove the columns themselves or change the data values; it only removes the unnecessary column-axis label.

The displayed rows confirm that the dataset pivot was executed successfully.

In [3]:
age_groups_and_sex_pivoted.shape

(510, 5)

The pivoted dataset contains 510 rows and 5 columns.
The rows represent every combination of seventeen age groups, three sex categories, and ten years.
The four columns now are `Vecuma grupa`, `Dzimums`, `Gads`, `Iebrauca Latvijā`, and `Izbrauca no Latvijas`.

In [4]:
age_groups_and_sex_pivoted["Migrācijas saldo"] = (
    age_groups_and_sex_pivoted["Iebrauca Latvijā"] - age_groups_and_sex_pivoted["Izbrauca no Latvijas"])

age_groups_and_sex_pivoted.head()

,Vecuma grupa,Dzimums,Gads,Iebrauca Latvijā,Izbrauca no Latvijas,Migrācijas saldo
0,0–4 gadi,Pavisam,2016,976,727,249
1,0–4 gadi,Pavisam,2017,747,677,70
2,0–4 gadi,Pavisam,2018,670,649,21
3,0–4 gadi,Pavisam,2019,655,511,144
4,0–4 gadi,Pavisam,2020,484,467,17


A new column, `Migrācijas saldo`, is calculated as immigration minus emigration: `Migrācijas saldo = Iebrauca Latvijā - Izbrauca no Latvijas`. A negative migration saldo means that more people left Latvia than arrived. A positive migration saldo means that more people arrived in Latvia than left.

In [5]:
age_groups_and_sex_pivoted.to_csv(
    "../data/processed/ibe030-migration-by-age-groups-and-sex-2016-2025-processed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("The dataset has been saved")

The dataset has been saved


The processed wide format dataset is saved to the `data/processed` folder for use in analysis as the next step in the planned methodology.
`index=False` prevents pandas from writing its numeric row index as an extra column in the `.csv` file. `encoding="utf-8-sig"` writes a UTF-8 file with a byte order mark, which helps some programs, including Microsoft Excel, to correctly recognize the encoding and display Latvian characters.